# 🔍 AI Evaluation Error Analysis Toolkit

## Overview

This notebook provides a comprehensive data analysis toolkit for AI generated data. It helps you understand patterns in data, including sommon evaluation failures, identify common issues, and gain insights into how to improve your AI system's performance.



## Prerequisites

Before running this notebook, ensure you have:
- Evaluation data in the expected format (see Data Structure section below)
- OpenAI/Azure OpenAI credentials configured for LLM analysis
- Required Python packages installed

Let's get started! 🚀

# 🛠️ Setup and Configuration

## Environment Setup

This notebook requires several dependencies and configuration steps:

### 1. Install Required Packages
```bash
pip install matplotlib seaborn wordcloud plotly ipywidgets
```

### 2. Configure OpenAI/Azure OpenAI (for LLM analysis)
Create a `.env` file with your credentials:
```env
AZURE_OPENAI_API_KEY=your_api_key_here
AZURE_OPENAI_ENDPOINT=https://your-endpoint.openai.azure.com/
AZURE_OPENAI_API_VERSION=2024-02-01
AZURE_OPENAI_DEPLOYMENT_NAME=your_deployment_name
```



### 3. Enable Auto-reload
The cell below enables automatic reloading of modules, so changes to the error analyzer code are picked up automatically.

In [86]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 📦 Import Analysis Toolkit

Now we'll import our error analysis tools:
- **`ErrorAnalyzer`**: The main class that performs error pattern analysis
- **`load_evaluation_data_with_samples`**: Utility to load evaluation data with full conversation context

**Note**: The enhanced report now includes conversation data directly in `raw_imperfect_data`, so the interactive drill-down no longer requires separate conversation data.

In [1]:
from data_analyzer import DataAnalyzer


# 📊 Load and Explore Your Data

## Data Loading

We'll load data that contains both:
1. **Context**: Scores, pass/fail results, and detailed reasons
2. **Original conversation**: Full conversation data with queries and responses
3. **Metadata**

## Expected Data Structure

Your evaluation data should follow this structure:
```python
{
    "context": this is the analysis context, for error analysis this is the Evaluation output,
    "conversation": the conversation this context is attached to,
    "metadata": metadata
}
```

Let's load your data and examine its structure:

In [4]:
# read the input from json
import json
with open("./data/data_analyzer_input.json", "r") as f:
    data_analyzer_input = json.load(f)

print(f"loaded {len(data_analyzer_input)} entries")
# print json with indentation
print(json.dumps(data_analyzer_input[0], indent=2))


loaded 292 entries
{
  "context": {
    "intent_resolution": 5.0,
    "intent_resolution_result": "pass",
    "intent_resolution_threshold": 3,
    "intent_resolution_reason": "The user asked for the distance from Sydney Opera House to Bondi Beach. The agent provided the straight-line distance in both kilometers and miles, directly addressing the request. The offer for further directions is helpful but not required. The intent is fully resolved."
  },
  "conversation": {
    "query": [
      {
        "role": "system",
        "content": "You are a WeatherAndMaps Assistant, an expert in providing weather information and location-based services using Azure Maps and Weather APIs.\n\nYou have access to the following powerful tools:\n\n**Weather Functions:**\n- azure_maps_weather_current_conditions: Get current weather conditions for any location\n- azure_maps_weather_daily_forecast: Get weather forecasts for up to 5 days\n- azure_maps_weather_hourly_forecast: Get hourly weather forecasts 

In [ ]:
# filter data_analyzer_input to entries with metadata/score less than 3
data_analyzer_input_error = [
    entry for entry in data_analyzer_input if entry.get("metadata", {}).get("score", 0) < 3
]
print(f"filtered {len(data_analyzer_input_error)} entries with score < 3")

filtered 36 entries with score < 3


## Run Analyzer

In [ ]:
data_analyzer = DataAnalyzer()

data_analysis_llm_results = data_analyzer.analyze(entries=data_analyzer_input_error, num_clusters=3, clustering_method="llm")

In [ ]:
# save report as json
import json
with open("./data/data_analysis_llm_report.json", "w") as f:
    json.dump(data_analysis_llm_results, f, indent=4)

In [100]:
data_analysis_llm_results.keys()

dict_keys(['summary', 'entries', 'subclusters', 'clusters', 'llm_analysis', 'axes', 'raw'])

In [101]:
data_analysis_results

{'summary': {'total_entries': 36,
  'unique_subcluster_labels': 35,
  'total_clusters': 3,
  'clustering_method': 'llm'},
 'entries': [{'id': 0,
   'context_summary': '{\'intent_resolution\': 2.0, \'intent_resolution_result\': \'fail\', \'intent_resolution_threshold\': 3, \'intent_resolution_reason\': "The user wanted to know how late restaurants stay open near Times Square. The agent did not provide any closing times or general information, instead asking for clarification. This does not resolve the user\'s intent and leaves their question unanswered."}',
   'conversation_summary': 'The conversation centers on a Weather and Maps Assistant designed to provide weather updates and location-based services using Azure Maps and Weather APIs. Key functions include weather forecasts, address and point-of-interest searches, geolocation, and timezone information, serving users seeking real-time weather and mapping data.',
   'combined_summary': 'Context: {\'intent_resolution\': 2.0, \'intent_re

# output format


### Data Analysis Report Structure

The object `data_analysis_results` is a dictionary with these top-level keys:

- `summary`: Overall stats
  - `total_entries`: Number of input entries analyzed
  - `unique_subcluster_labels`: Count of distinct subclusters
  - `total_clusters`: Number of top-level clusters
  - `clustering_method`: `llm` or `classic`
- `axes`: Names of the 2D coordinate axes (always `embeddings1`, `embeddings2`)
- `entries`: List of per-entry records
  - Each entry contains:
    - `id`: Entry identifier (original id or index)
    - `context_summary`: Context-focused summary text (or raw context fallback)
    - `conversation_summary`: Conversation summary (may be empty if no conversation)
    - `combined_summary`: Joined context + conversation summary
    - `subcluster_label`: Assigned subcluster label
    - `metadata`: Original metadata plus:
      - `conversation_turns`: Count of detected conversation turns
    - `coordinates`: 2D embedding coordinates `[x, y]`
- `subclusters`: Mapping subcluster_label -> object
  - `entry_ids`: List of entry ids in the subcluster
  - `count`: Number of entries
  - `coordinates`: Mean 2D position of its entries
- `clusters`: Mapping cluster_name -> object
  - `weight`: Total entries in all member subclusters
  - `subcluster_labels`: List of subcluster labels in the cluster
  - `subcluster_counts`: Per-subcluster entry counts
  - `description`: Textual description (LLM / heuristic / classic)
  - `coordinates`: Weighted average (by subcluster count) 2D position
- `llm_analysis`: Optional natural language bullet summary (None if disabled or no LLM)
- `classic_metadata`: Present only for `classic` method (e.g., vectorizer, subcluster_k)
- `raw`: The original input entries you supplied (unaltered)

#### Drill-Down Flow
1. Start at `clusters` to view thematic groups.
2. Expand a cluster to see its `subcluster_labels`.
3. Look up each label in `subclusters` to get member entry ids.
4. Fetch those entries from `entries` for detailed summaries & metadata.

#### Coordinates
- Subcluster and cluster coordinates are averages (cluster average is weighted by subcluster size).

Use these fields to build dashboards, interactive scatter plots, or hierarchical explorers.

# 📈 2D Cluster Visualizations

Below we visualize the embedding / coordinate space produced by the DataAnalyzer.

- Subcluster view: Each point represents a subcluster. Point size = number of entries.
- Entry view: Each point is an entry colored by its subcluster.

Axis labels come from `data_analysis_results['axes']`.

In [ ]:
from utils import visualize_data_analyzer_2d

visualize_data_analyzer_2d(data_analysis_llm_results)

# Test Classic Method

In [ ]:
data_analyzer = DataAnalyzer(
)
data_analysis_classic_results = data_analyzer.analyze(entries=data_analyzer_input_error, num_clusters=3, clustering_method="classic")

In [31]:
import json
with open("data_analysis_classic_report.json", "w") as f:
    json.dump(data_analysis_classic_results, f)

In [32]:
data_analysis_classic_results

{'summary': {'total_entries': 36,
  'unique_subcluster_labels': 6,
  'total_clusters': 3,
  'clustering_method': 'classic'},
 'entries': [{'id': 0,
   'context_summary': '{\'intent_resolution\': 2.0, \'intent_resolution_result\': \'fail\', \'intent_resolution_threshold\': 3, \'intent_resolution_reason\': "The user wanted to know how late restaurants stay open near Times Square. The agent did not provide any closing times or general information, instead asking for clarification. This does not resolve the user\'s intent and leaves their question unanswered."}',
   'conversation_summary': 'The conversation outlines the role and capabilities of a WeatherAndMaps Assistant, which provides weather information and location-based services using Azure Maps and Weather APIs. Key entities include weather data functions, location and search tools, and utility features, all designed to deliver real-time weather, forecasts, and geographic information for various user queries.',
   'combined_summary':

In [19]:
from utils import visualize_data_analyzer_2d

visualize_data_analyzer_2d(data_analysis_classic_results)